# Customer Churn Prediction

### Project Objective

Analyze customer data to identify factors associated with customer churn, perform data cleaning and feature engineering, and build a machine learning model to predict customer churn.

## Task 5 – Data Loading and SQL Connectivity

In [1]:
import pandas as pd
import pyodbc
print("Pandas version:", pd.__version__)
print("pyodbc imported successfully")

Pandas version: 3.0.3
pyodbc imported successfully


In [2]:
connection_string = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=.\\SQLEXPRESS;"
    "Database= ChurnDB;"  
    "Trusted_Connection=yes;"
)
conn = pyodbc.connect(connection_string)
print("Connected to ChurnDB successfully!")

Connected to ChurnDB successfully!


In [3]:
query = "SELECT * FROM [dbo].[vw_ChurnData]"
df = pd.read_sql(query, conn)

C:\Users\nandhini\AppData\Local\Temp\ipykernel_10580\1028094426.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,None,DSL,False,...,False,False,False,False,Month-to-month,True,Electronic check,29.850000,29.85,False
1,5575-GNVDE,Male,False,False,False,34,True,False,DSL,True,...,True,False,False,False,One year,False,Mailed check,56.950001,1889.50,False
2,3668-QPYBK,Male,False,False,False,2,True,False,DSL,True,...,False,False,False,False,Month-to-month,True,Mailed check,53.849998,108.15,True
3,7795-CFOCW,Male,False,False,False,45,False,None,DSL,True,...,True,True,False,False,One year,False,Bank transfer (automatic),42.299999,1840.75,False
4,9237-HQITU,Female,False,False,False,2,True,False,Fiber optic,False,...,False,False,False,False,Month-to-month,True,Electronic check,70.699997,151.65,True


### Data Inspection

In [26]:
df.shape

(7043, 21)

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   bool   
 3   Partner           7043 non-null   bool   
 4   Dependents        7043 non-null   bool   
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   bool   
 7   MultipleLines     6361 non-null   object 
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    5517 non-null   object 
 10  OnlineBackup      5517 non-null   object 
 11  DeviceProtection  5517 non-null   object 
 12  TechSupport       5517 non-null   object 
 13  StreamingTV       5517 non-null   object 
 14  StreamingMovies   5517 non-null   object 
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   bool   
 17  Paymen

In [34]:
# Unique Values
df.nunique()

customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          2
InternetService        3
OnlineSecurity         2
OnlineBackup           2
DeviceProtection       2
TechSupport            2
StreamingTV            2
StreamingMovies        2
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6530
Churn                  2
dtype: int64

### Missing Value Check

In [29]:
df.isnull().sum(axis=0)

customerID             0
gender                 0
SeniorCitizen          0
Partner                0
Dependents             0
tenure                 0
PhoneService           0
MultipleLines        682
InternetService        0
OnlineSecurity      1526
OnlineBackup        1526
DeviceProtection    1526
TechSupport         1526
StreamingTV         1526
StreamingMovies     1526
Contract               0
PaperlessBilling       0
PaymentMethod          0
MonthlyCharges         0
TotalCharges          11
Churn                  0
dtype: int64

In [40]:
import numpy as np
np.round((df.isnull().sum(axis=0) / len(df)) * 100, 2)

customerID           0.00
gender               0.00
SeniorCitizen        0.00
Partner              0.00
Dependents           0.00
tenure               0.00
PhoneService         0.00
MultipleLines        9.68
InternetService      0.00
OnlineSecurity      21.67
OnlineBackup        21.67
DeviceProtection    21.67
TechSupport         21.67
StreamingTV         21.67
StreamingMovies     21.67
Contract             0.00
PaperlessBilling     0.00
PaymentMethod        0.00
MonthlyCharges       0.00
TotalCharges         0.16
Churn                0.00
dtype: float64

### Observation:

**Eight** columns have missing values. **MultipleLines** has **9.68%** missing values, **six internet-related service columns have 21.67%** missing values each, and **TotalCharges** has only **0.16%** missing values. The missing values in the service columns are related to customers who do not have the corresponding service.

### Missing Value Treatment:

The missing values in the **internet-related service** columns will be treated as **"No internet service"**, because these customers do not have internet service. Similarly, missing values in **MultipleLines** will be as **"No phone service"** for customers who do not have phone service. The few missing **TotalCharges** values will be handled separately after checking their tenure and related billing information.

### Duplicate Records Check

In [35]:
df.duplicated().sum()

np.int64(0)

There are no duplicate records.

### Descriptive Statistics

In [36]:
df.describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7032.000000
mean,32.371149,64.761692,2283.300441
std,24.559481,30.090047,2266.771362
min,0.000000,18.250000,18.800000
25%,9.000000,35.500000,401.450000
50%,29.000000,70.349998,1397.475000
75%,55.000000,89.849998,3794.737500
max,72.000000,118.750000,8684.800000


In [37]:
df.describe(include='O')

C:\Users\nandhini\AppData\Local\Temp\ipykernel_4856\2537740217.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='O')


,customerID,gender,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaymentMethod
count,7043,7043,6361,7043,5517,5517,5517,5517,5517,5517,7043,7043
unique,7043,2,2,3,2,2,2,2,2,2,3,4
top,7590-VHVEG,Male,False,Fiber optic,False,False,False,False,False,False,Month-to-month,Electronic check
freq,1,3555,3390,3096,3498,3088,3095,3473,2810,2785,3875,2365


### Key observations
*  **Tenure:** ranges from **0 to 72 months**. The median is 29 months, meaning half of the customers have been with the company for less than about 29 months.
*  **MonthlyCharges:** ranges from 18.25 to 118.75. The median monthly charge is about 70.35.
*  **TotalCharges:** ranges from 18.80 to 8,684.80. There are 7,032 non-null values out of 7,043, so 11 values are missing. This is something we handled during Task 6.
*  There are **3** different contract types, with **Month-to-month** being the most common.
*  The dataset includes **4 payment methods**, with **Electronic check** being the most commonly used by customers.

## Task 6 – Data Cleaning and Feature Preparation

### Handling Missing Values

In [6]:
df['MultipleLines'] = df['MultipleLines'].fillna("No phone service")

columns = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',  'TechSupport', 'StreamingTV', 'StreamingMovies']  
for col in columns:
    df[col] = df[col].fillna("No internet service")

In [7]:
df[df['TotalCharges'].isnull()][['tenure', 'MonthlyCharges', 'TotalCharges']]

,tenure,MonthlyCharges,TotalCharges
488,0,52.549999,NaN
753,0,20.250000,NaN
936,0,80.849998,NaN
1082,0,25.750000,NaN
1340,0,56.049999,NaN
3331,0,19.850000,NaN
3826,0,25.350000,NaN
4380,0,20.000000,NaN
5218,0,19.700001,NaN
6670,0,73.349998,NaN


The **11 missing TotalCharges** values belong to customers with zero tenure. Since these customers have just started their service, the missing total charge is treated as **0** rather than using a statistical value such as the mean or median.

In [8]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [9]:
df.isnull().sum(axis=0)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## Feature Preparation

In [10]:
df['gender'] = df['gender'].map({'Male':1, 'Female':0})

In [11]:
bool_col = ['SeniorCitizen', 'Partner', 'Dependents','PhoneService', 'PaperlessBilling', 'Churn']
df[bool_col] = df[bool_col].astype(int)

In [12]:
categorical_cols = [
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaymentMethod'
]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True,  dtype=int)

In [14]:
df.dtypes

customerID                                   str
gender                                     int64
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
MultipleLines_True                         int64
MultipleLines_No phone service             int64
InternetService_Fiber optic                int64
InternetService_No                         int64
OnlineSecurity_True                        int64
OnlineSecurity_No internet service         int64
OnlineBackup_True                          int64
OnlineBackup_No internet service           int64
DeviceProtection_True                      int64
DeviceProtection_No 

In [5]:
# Remove irrelevant customer ID column
df = df.drop(columns=['customerID'])

In [16]:
# Save the cleaned dataset as a CSV file
df.to_csv('cleaned_churn_data.csv', index=False)

In [17]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_No internet service,StreamingTV_True,StreamingTV_No internet service,StreamingMovies_True,StreamingMovies_No internet service,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.850000,29.85,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.950001,1889.50,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.849998,108.15,1,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.299999,1840.75,0,...,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70.699997,151.65,1,...,0,0,0,0,0,0,0,0,1,0


## Task 8 – Feature Engineering

In [23]:
df['tenure'].min(), df['tenure'].max()

(np.int64(0), np.int64(72))

In [28]:
service_columns = [
    'OnlineSecurity_True',
    'OnlineBackup_True',
    'DeviceProtection_True',
    'TechSupport_True',
    'StreamingTV_True',
    'StreamingMovies_True'
]
df['TotalServiceUsed'] = df[service_columns].sum(axis=1)

In [32]:
df['TotalServiceUsed'].value_counts().sort_index()

TotalServiceUsed
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64

In [33]:
df[['TotalServiceUsed', 'Churn']].corr()

,TotalServiceUsed,Churn
TotalServiceUsed,1.000000,-0.087698
Churn,-0.087698,1.000000


**TotalServicesUsed** has a **weak negative** correlation with churn **(-0.0877)**. This means customers who use more services are slightly less likely to churn, but it does not strongly say that they will not churn.

In [39]:
df['AvgMonthlySpend'] = 0.0

mask = df['tenure'] > 0

df.loc[mask, 'AvgMonthlySpend'] = (
    df.loc[mask, 'TotalCharges'] / df.loc[mask, 'tenure']
)

In [40]:
df[['tenure', 'TotalCharges', 'AvgMonthlySpend']].head(10)

,tenure,TotalCharges,AvgMonthlySpend
0,1,29.85,29.850000
1,34,1889.50,55.573529
2,2,108.15,54.075000
3,45,1840.75,40.905556
4,2,151.65,75.825000
5,8,820.50,102.562500
6,22,1949.40,88.609091
7,10,301.90,30.190000
8,28,3046.05,108.787500
9,62,3487.95,56.257258


In [41]:
df[['AvgMonthlySpend', 'Churn']].corr()

,AvgMonthlySpend,Churn
AvgMonthlySpend,1.000000,0.193301
Churn,0.193301,1.000000


**AvgMonthlySpend** has a **weak positive correlation** with churn **(0.1933)**. This meansCustomers with higher average monthly spending are slightly more likely to churn, , but the relationship is not strong.

In [42]:
def customer_segment(x):
    if x <= 12:
        return "New"
    elif x <= 36:
        return "Established"
    else:
        return "Loyal"

In [43]:
df['CustomerSegment'] = df['tenure'].apply(customer_segment)

In [44]:
df['CustomerSegment'].value_counts()

CustomerSegment
Loyal          3001
New            2186
Established    1856
Name: count, dtype: int64

In [45]:
df.groupby('CustomerSegment')['Churn'].mean()

CustomerSegment
Established    0.255388
Loyal          0.119294
New            0.474382
Name: Churn, dtype: float64

CustomerSegment is a useful feature because **New customers** have a **higher churn rate**, while **Loyal customers** have a **lower churn rate**. This shows that customer tenure is related to churn.

In [49]:
df[['TotalServiceUsed', 'AvgMonthlySpend', 'CustomerSegment']].head()

,TotalServiceUsed,AvgMonthlySpend,CustomerSegment
0,1,29.850000,New
1,2,55.573529,Established
2,2,54.075000,New
3,3,40.905556,Loyal
4,0,75.825000,New


In [50]:
df.shape

(7043, 34)

In [51]:
segment_mapping = {
    'New': 0,
    'Established': 1,
    'Loyal': 2
}

df['CustomerSegment'] = df['CustomerSegment'].map(segment_mapping)

## Task 9 – Baseline Model: Logistic Regression

In [53]:
# Define input features (X) and target variable (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

### Train/test split

In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [57]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5634, 33), (1409, 33), (5634,), (1409,))

In [59]:
# Train the model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

C:\Users\nandhini\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [60]:
# Make predictions
y_pred = model.predict(X_test)

In [61]:
# predictions on 5 test customers
y_pred[:5]

array([0, 1, 0, 0, 0])

In [62]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8062455642299503


Logistic Regression achieved an accuracy of **80.62%** on the test data. The model correctly predicted the churn status of approximately **81% of customers**.